In [ ]:
import numpy as np
import math
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ============================================================
# 1. INPUT DATA (your provided data)
# ============================================================

X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.7786275 ],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.9265642 ],
    [0.926564  , 1.026564  ],
    [0.747504  , 0.20897   ],
    [0.683406  , 0.063769  ],
    [0.583137  , 0.012549  ]
])

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726
])

# Clip inputs to respect domain [0,1]
X = np.clip(X, 0, 1)

# ============================================================
# 2. MODEL SELECTION: Deep Ensemble of Shallow MLPs
# ============================================================

def make_model(seed):
    """
    Shallow NN chosen intentionally because dataset is VERY small.
    Ensemble provides uncertainty for EI/PI.
    """
    return Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(32, 16),
            activation='relu',
            solver='adam',
            alpha=0.001,              # L2 penalty for stability
            learning_rate='adaptive',
            learning_rate_init=0.01,
            max_iter=3000,
            tol=1e-7,
            random_state=seed
        ))
    ])

def fit_ensemble(X, y, n=9):
    models = []
    for i in range(n):
        model = make_model(202 + i)
        model.fit(X, y)
        models.append(model)
    return models

def ensemble_predict(models, Xcand):
    preds = np.vstack([m.predict(Xcand) for m in models])
    mu = preds.mean(axis=0)
    std = preds.std(axis=0, ddof=1) + 1e-9
    return mu, std

def erf_vec(x):
    return np.vectorize(math.erf)(x)

def compute_pi_ei(mu, std, y_best, xi=0.01):
    z = (mu - y_best - xi) / std
    pdf = (1/np.sqrt(2*np.pi)) * np.exp(-0.5 * z*z)
    cdf = 0.5 * (1 + erf_vec(z / np.sqrt(2)))
    ei = (mu - y_best - xi) * cdf + std * pdf
    ei[std <= 0] = 0
    return cdf, ei


# ============================================================
# 3. TRAIN ENSEMBLE
# ============================================================

models = fit_ensemble(X, y, n=9)

idx_best = np.argmax(y)
x_best = X[idx_best]
y_best = y[idx_best]


# ============================================================
# 4. GENERATE CANDIDATE POINTS WITH DOMAIN = [0,1]×[0,1]
# ============================================================

rng = np.random.default_rng(999)

N = 20000
Xcand = rng.uniform(0, 1, size=(N, 2))

# Add local refinement near best point
local_band = rng.normal(loc=x_best, scale=0.05, size=(N//6, 2))
local_band = np.clip(local_band, 0, 1)
Xcand = np.vstack([Xcand, local_band])


# ============================================================
# 5. COMPUTE EI + PI
# ============================================================

mu, std = ensemble_predict(models, Xcand)
pi, ei = compute_pi_ei(mu, std, y_best=y_best)

# Best next point
k = int(np.argmax(ei))
x_next = Xcand[k]
mu_next = mu[k]
std_next = std[k]
pi_next = pi[k]
ei_next = ei[k]


# ============================================================
# 6. PRINT RESULTS
# ============================================================

print("================================================")
print("CURRENT BEST OBSERVED POINT")
print("================================================")
print(f"x_best = {x_best},   y_best = {y_best:.6f}\n")

print("================================================")
print("RECOMMENDED NEXT QUERY POINT (EI)")
print("================================================")
print(f"x_next = {x_next}")
print(f"Predicted mean μ = {mu_next:.6f}")
print(f"Predictive std σ = {std_next:.6f}")
print(f"Probability of Improvement (PI) = {pi_next:.3f}")
print(f"Expected Improvement (EI) = {ei_next:.6f}\n")

print("================================================")
print("DETAILED REASONING")
print("================================================")
print(f"""
• Domain is restricted to [0,1], so all candidate points are valid.
• The surrogate model is a deep ensemble of small neural networks,
  which is the best method for this small dataset because:
    - Small NNs avoid overfitting
    - Ensemble variance gives good exploration signals
    - Smooth function fitting works well in 2D

• The chosen x_next has:
      μ = {mu_next:.6f}
      σ = {std_next:.6f}
      PI = {pi_next:.3f}  (~{pi_next*100:.1f}% chance to beat y*)
      EI = {ei_next:.6f}

• This point lies inside the valid range [0,1] and yields the highest EI
  among 23,000 candidate points sampled.

• Because EI rewards “high mean + uncertainty”, this point gives the best
  chance to find an output *greater* than the current max y* = {y_best:.6f}.
""")
